# 沪深 A 股股票筛选 · 主流程 Notebook

> 本 Notebook 是项目的**交互式入口**。所有核心逻辑在 `src/` 包内,本文件按步骤调用并展示中间结果。

## 项目目标

按照三个 Skill 自动筛选优质股票:

1. **Skill 1 · 成长性筛选**  
   过去 3 个完整年报(2023/2024/2025)中,4 项核心指标(每股收益、主营业务收入、毛利率、净利润)的**同比增长率连续两年(2024、2025)≥ 35%** 的股票入选。  
   若 2026 年一季报已发布,附加展示(不参与硬筛)。

2. **Skill 2 · 板块归类**  
   将 Skill1 入选的股票按东方财富行业板块归类。

3. **Skill 3 · 同板块对比**  
   在每个板块内对 4 项指标排名,**至少 2 项进入板块前 50%** 的股票入选。  

4. **Skill 4 · 周K 线技术共振(可选)**  
   对 Skill3 入选的股票叠加周线技术面验证(基准:最近一根已收盘的完整周K):  
   - 该周阳线  
   - 该周成交量较上一周增幅 ≥ 70%  
   - 周线 MA5 > MA10 > MA20 > MA30 > MA60  
   - 周收盘价 > MA5  
   通过 Skill4 的股票被标记为"⭐ 技术面共振"。

最终按"领先项数 → 综合得分"降序输出 Markdown 报告。

## 数据源

- **akshare**(免费、不需注册)— 提供股票列表、财务摘要、行业板块、行情数据。

## 运行须知

- 首次运行需联网拉取财报,**全市场约 5400 只股票预计耗时 30-90 分钟**(取决于网络)。
- 数据默认缓存到 `data/cache/` 目录,7 天内不会重复拉取。
- 强烈建议**先用 `sample_size=50` 跑通**全流程,再放开全市场。

## 1 · 环境准备

如未安装依赖,先在 terminal 执行:

```bash
pip install akshare pandas numpy openpyxl jupyter
```

In [8]:
# 让 notebook 能找到 src 包
import sys, os
from pathlib import Path
ROOT = Path(os.getcwd()).resolve().parent if Path(os.getcwd()).name == 'notebooks' else Path(os.getcwd()).resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

print('项目根目录:', ROOT)


项目根目录: D:\stock_screener_proj


## 2 · 加载配置

所有可调参数集中在 `src/config.py`,可以在这里临时覆盖:

In [9]:
from src import config

# 例:把 Skill1 的同比增长阈值从 35% 临时改为 30%(本次运行生效)
# config.SKILL1_MIN_GROWTH = 0.15

print('Skill1 阈值:', f"4项指标同比增长率 ≥ {config.SKILL1_MIN_GROWTH*100:.0f}%")
print('Skill1 必需年份:', config.SKILL1_REQUIRED_YEARS)
print('Skill3 至少领先项数:', config.SKILL3_LEADING_THRESHOLD)
print('Skill3 板块前百分比:', f"{config.SKILL3_TOP_PERCENTILE*100:.0f}%")
print('Skill4 启用:', config.SKILL4_ENABLED, f"(周量增 ≥ {config.SKILL4_MIN_VOL_GROWTH*100:.0f}%)")


Skill1 阈值: 4项指标同比增长率 ≥ 25%
Skill1 必需年份: [2023, 2024, 2025]
Skill3 至少领先项数: 2
Skill3 板块前百分比: 50%
Skill4 启用: True (周量增 ≥ 70%)


## 3 · 拉取股票列表(沪深 A 股)

实时行情接口同时返回名称、最新价、PE、PB、市值、换手率,我们顺便缓存下来。

In [10]:
from src import data_fetcher

stocks = data_fetcher.get_stock_list()
print(f'股票总数(已剔除北交所):{len(stocks)}')
print(f'其中 ST 股票:{stocks["is_st"].sum()}')
stocks.head(10)


股票总数(已剔除北交所):5840
其中 ST 股票:306


,code,name,market,is_st
0,920156,N海昌,其他,False
1,688381,帝奥微,上海,False
2,688221,前沿生物-U,上海,False
3,300029,*ST天龙,深圳,True
4,688530,欧莱新材,上海,False
5,688677,海泰新光,上海,False
6,688001,华兴源创,上海,False
7,301248,杰创智能,深圳,False
8,688184,ST帕瓦,上海,True
9,301500,飞南资源,深圳,False


## 4 · 调试模式:小样本跑通流水线

> 全市场拉财报较慢,建议先用 50 只验证逻辑没问题,再放开全市场。

下面这一格会:
1. 抽前 N 只股票
2. 串行拉取财报(带本地缓存)
3. 跑 Skill1 / Skill2 / Skill3
4. 在 `reports/` 目录生成 Markdown 报告

In [11]:
from src import pipeline

result = pipeline.run_pipeline(sample_size=500)

print()
print('=' * 60)
print(f'Skill1 入选: {len(result["qualified"])} 只')
print(f'Skill3 最终入选: {int(result["ranked"]["is_winner"].sum()) if not result["ranked"].empty else 0} 只')
print(f'报告路径: {result["report_path"]}')


2026-04-28 10:53:18,953 | INFO | [1/7] 加载沪深 A 股列表
2026-04-28 10:53:18,963 | INFO | 调试模式:仅处理前 500 只
2026-04-28 10:53:18,964 | INFO | 共 500 只候选股票
2026-04-28 10:53:18,964 | INFO | [2/7] 批量拉取财报(使用本地缓存)
2026-04-28 10:53:19,016 | INFO | 财务数据进度 50/500
2026-04-28 10:53:19,069 | INFO | 财务数据进度 100/500
2026-04-28 10:53:19,121 | INFO | 财务数据进度 150/500
2026-04-28 10:53:19,217 | INFO | 财务数据进度 200/500
2026-04-28 10:53:19,315 | INFO | 财务数据进度 250/500
2026-04-28 10:53:19,413 | INFO | 财务数据进度 300/500
2026-04-28 10:53:19,511 | INFO | 财务数据进度 350/500
2026-04-28 10:53:19,609 | INFO | 财务数据进度 400/500
2026-04-28 10:53:19,708 | INFO | 财务数据进度 450/500
2026-04-28 10:53:19,811 | INFO | 财务数据进度 500/500
2026-04-28 10:53:19,837 | INFO | 财报记录:24398 行,覆盖 500 只股票
2026-04-28 10:53:19,839 | INFO | [3/7] Skill1 同比增长筛选
2026-04-28 10:53:20,225 | INFO | Skill1 入选:8 只
2026-04-28 10:53:20,225 | INFO | [4/7] Skill2 板块归类
2026-04-28 10:53:20,230 | INFO | 使用板块缓存(共 5591 只股票,207 个行业)
2026-04-28 10:53:20,235 | INFO | Skill2 板块分布:
   sector


Skill1 入选: 8 只
Skill3 最终入选: 3 只
报告路径: D:\stock_screener_proj\reports\screening_report_20260428_105320.md


## 5 · 查看 Skill1 入选结果

4 个指标的逐年同比增长率明细:

In [ ]:
qualified = result['qualified']
if qualified.empty:
    print('Skill1 没有入选股票,可调小 SKILL1_MIN_GROWTH 阈值再试')
else:
    qualified.head(20)


In [ ]:
# 按板块统计入选数量
from src import skill2_sector
if not qualified.empty:
    summary = skill2_sector.sector_summary(qualified)
    print(summary.head(20).to_string(index=False))


## 6 · 查看 Skill3 板块内对比结果

In [ ]:
ranked = result['ranked']
if not ranked.empty:
    cols = ['code','sector','leading_count','composite_score','is_winner',
            'eps','revenue','gross_margin','net_profit']
    cols = [c for c in cols if c in ranked.columns]
    ranked[cols].head(30)


In [ ]:
# 仅看最终入选(is_winner=True)
if not ranked.empty:
    winners = ranked[ranked['is_winner']].copy()
    print(f'最终入选 {len(winners)} 只')
    winners.head(20)


## 6.5 · ⭐ Skill4 周K线技术共振结果

In [ ]:
# 同时通过 Skill3 + Skill4 的股票(技术面共振,含金量最高)
if not ranked.empty and 'skill4_pass' in ranked.columns:
    sk4_winners = ranked[ranked['is_winner'] & ranked['skill4_pass']].copy()
    print(f'技术面共振股票:{len(sk4_winners)} 只')
    show_cols = ['code','sector','last_close','vol_growth','ma5','ma10','ma20','ma30','ma60',
                 'is_yang','vol_passed','ma_aligned','close_above_ma5']
    show_cols = [c for c in show_cols if c in sk4_winners.columns]
    sk4_winners[show_cols].head(20)


## 7 · 阅读 Markdown 报告

报告路径在 `result['report_path']`,直接在 Notebook 内渲染:

In [ ]:
from IPython.display import Markdown, display
if result['report_path']:
    md_text = Path(result['report_path']).read_text(encoding='utf-8')
    # 报告可能很长,只展示前 5000 字符
    display(Markdown(md_text[:5000] + ('\n\n*(下略,完整内容请打开报告文件)*' if len(md_text) > 5000 else '')))


## 8 · 全市场运行(确认无误后再执行)

⚠️ **首次运行约 30-90 分钟**(取决于网络与缓存)。
重复运行会复用 `data/cache/` 内的财报缓存,耗时大幅降低。

In [ ]:
# 取消下一行注释以全市场运行
# result_full = pipeline.run_pipeline(sample_size=None)


## 9 · 进阶:叠加辅助筛选(可选)

`src/auxiliary_filters.py` 提供基本面 / 技术面 / 资金面 / 板块过滤,可以在 Skill3 结果之上再做一轮过滤。

In [ ]:
from src import auxiliary_filters

# 例:在 Skill3 入选基础上,叠加 PE<50、市值>50亿 的过滤
if not result['ranked'].empty:
    snapshot = auxiliary_filters.get_fundamental_snapshot()
    winners = result['ranked'][result['ranked']['is_winner']].copy()
    winners = winners.merge(snapshot[['code','pe_ttm','pb','mkt_cap']], on='code', how='left')
    
    # 总市值 akshare 返回单位是元
    final = auxiliary_filters.filter_fundamentals(
        winners,
        pe_max=50,
        mkt_cap_min=50e8,  # 50 亿
    )
    print(f'叠加 PE<50 & 市值>50亿 后剩余:{len(final)} 只')
    final.head(20)


---

### 自定义筛选

如果你想改变筛选规则,直接修改 `src/config.py`:

| 参数 | 含义 | 默认 |
| --- | --- | --- |
| `SKILL1_MIN_GROWTH` | Skill1 同比增长率下限 | `0.35` |
| `SKILL1_REQUIRED_YEARS` | 必需有完整年报的年份 | `[2023,2024,2025]` |
| `SKILL3_LEADING_THRESHOLD` | Skill3 至少领先指标数 | `2` |
| `SKILL3_TOP_PERCENTILE` | Skill3 板块内前 X% 算"领先" | `0.5` |
| `SKILL3_MIN_PEERS` | 同板块至少多少只才参与对比 | `3` |
| `SKILL4_ENABLED` | 是否启用 Skill4 周K线技术筛选 | `True` |
| `SKILL4_MIN_VOL_GROWTH` | Skill4 周成交量同比增幅下限 | `0.70`(70%) |
| `EXCLUDE_ST` | 是否排除 ST/*ST | `True` |
| `EXCLUDE_BJ` | 是否排除北交所 | `True` |

### 直接命令行运行(不通过 Notebook)

```bash
# 全市场
python -m src.pipeline

# 只跑前 100 只
python -m src.pipeline --sample 100
```
